# 184. LATS：怎样用 MCTS 统一 Agent 的推理、行动与规划？

> **面试问题：Selection、Expansion、Simulation、Backpropagation 如何落到工具环境？为什么不能在真实副作用上随意分支？**

## 先给结论

LATS 把语言模型提案、环境反馈、价值评估和反思放进 MCTS。每个节点必须对应可恢复的环境 state；UCT/PUCT 在已知高价值和未探索动作间权衡。搜索分支只应运行在模拟器、快照或只读工具中，最终选定路径才提交真实副作用。

## 推荐回答主线

1. 定义可哈希 state、合法 action、确定/随机 transition、终态 reward 和环境 snapshot。
2. 实现 PUCT selection、policy-prior expansion、rollout/value 与路径 backprop，检查访问次数守恒。
3. 加入 cycle/transposition、失败 reflection、深度/节点/tool/token 预算与不可行动作剪枝。
4. 只 commit 选中计划；比较 ReAct/ToT baseline 的 success、环境步数、成本和搜索方差。

## 教学实现边界

使用可逆整数环境而非真实网页/文件；启发式 prior/value 代替 LLM。它展示 MCTS 状态语义，不证明 LATS 在开放任务上的收益。

## 一手资料

- [Language Agent Tree Search](https://arxiv.org/abs/2310.04406)
- [ReAct](https://arxiv.org/abs/2210.03629)
- [AlphaZero / MCTS](https://arxiv.org/abs/1712.01815)


In [ ]:
import hashlib
import json
import math
from dataclasses import asdict, dataclass, field

import numpy as np

# 可逆玩具环境：从 1 通过 +1/+2/*2 到 7，越界或超步失败。
ACTIONS = ("+1", "+2", "*2")
TARGET, MAX_VALUE, MAX_STEPS = 7, 12, 5

@dataclass(frozen=True)
class State:
    value: int
    steps: int

assert TARGET < MAX_VALUE
assert len(ACTIONS) == 3
assert State(1, 0) != State(1, 1)


## 1. 环境 transition：终态由 state 推导，且不可再次执行

`done` 不能由调用者随手塞进节点，否则同一 state 可能同时被当成运行态和终态。这里用纯函数从 `(value, steps)` 推导 success、overflow 与 step budget；`transition` 在执行前先拒绝所有终态，保证终止语义只有一个事实来源。


In [ ]:
def state_key(state):
    return (state.value, state.steps)

def terminal_outcome(state):
    if state.value == TARGET:
        return True, 1.0, "success"
    if state.value > MAX_VALUE:
        return True, -1.0, "overflow"
    if state.steps >= MAX_STEPS:
        return True, -1.0, "step_budget"
    return False, 0.0, "running"

def transition(state, action):
    terminal, _, reason = terminal_outcome(state)
    if terminal:
        raise RuntimeError(f"终态不可继续 transition: {reason}")
    if action not in ACTIONS:
        raise ValueError("非法动作")
    value = state.value + 1 if action == "+1" else state.value + 2 if action == "+2" else state.value * 2
    next_state = State(value, state.steps + 1)
    done, reward, _ = terminal_outcome(next_state)
    return next_state, reward, done

# 转移保持旧 state 不变；success、overflow 与耗尽步数完全由新 state 推导。
start = State(1, 0)
next_state, reward, done = transition(start, "*2")
assert start == State(1, 0) and next_state == State(2, 1)
assert transition(State(6, 2), "+1")[1:] == (1.0, True)
assert transition(State(11, 1), "*2")[1:] == (-1.0, True)
assert transition(State(5, MAX_STEPS - 1), "+1")[1:] == (-1.0, True)

# 反例：无论成功、越界还是耗尽步数，终态都不可继续产生后继。
blocked_terminal_transitions = 0
for terminal_state in (State(TARGET, 2), State(MAX_VALUE + 1, 1), State(6, MAX_STEPS)):
    try:
        transition(terminal_state, "+1")
    except RuntimeError:
        blocked_terminal_transitions += 1
assert blocked_terminal_transitions == 3


## 2. Node、共享统计与 PUCT：边属于树，统计属于 state

不同 action 或父节点可能到达相同 `(value, steps)`。Node 仍保存 parent/action 以还原路径，但 `visits/value_sum` 通过 `SharedStats` 引用 transposition table；terminal/reward 则每次从不可变 state 推导。PUCT 的探索常数必须是有限正数。


In [ ]:
@dataclass
class SharedStats:
    visits: int = 0
    value_sum: float = 0.0

    @property
    def q(self):
        return self.value_sum / self.visits if self.visits else 0.0

@dataclass
class Node:
    state: State
    parent: object = None
    action: str | None = None
    prior: float = 1.0
    stats: SharedStats = field(default_factory=SharedStats)
    children: dict = field(default_factory=dict)

    @property
    def terminal(self):
        return terminal_outcome(self.state)[0]

    @property
    def reward(self):
        return terminal_outcome(self.state)[1]

    @property
    def visits(self):
        return self.stats.visits

    @property
    def q(self):
        return self.stats.q

def validate_exploration(c):
    if isinstance(c, bool) or not isinstance(c, (int, float)) or not math.isfinite(c) or c <= 0:
        raise ValueError("PUCT exploration c 必须为有限正数")

def puct(parent, child, c=1.5):
    validate_exploration(c)
    return child.q + c * child.prior * math.sqrt(max(parent.visits, 1)) / (1 + child.visits)

# 两条边引用同一统计对象时，一侧更新会立即反映到另一侧的 q/visits。
root = Node(start)
shared = SharedStats()
child_a = Node(next_state, root, "*2", prior=0.7, stats=shared)
child_b = Node(State(2, 1), root, "+1", prior=0.2, stats=shared)
assert child_a.stats is child_b.stats and child_a.q == 0
shared.visits, shared.value_sum = 2, 1.0
assert child_a.q == child_b.q == 0.5 and child_b.visits == 2
assert puct(root, Node(State(3, 1), root, prior=0.8)) > puct(root, Node(State(3, 1), root, prior=0.1))

invalid_c_rejected = 0
for invalid_c in (0, -1.0, float("nan"), float("inf"), True):
    try:
        puct(root, child_a, invalid_c)
    except ValueError:
        invalid_c_rejected += 1
assert invalid_c_rejected == 5


## 3. Expansion：policy prior、transposition table 与 forbidden 同时生效

Expansion 只为当前仍允许的 action 建边；每个后继先用 `state_key(value, steps)` 查询共享统计。这样两条同状态边不是“数值碰巧相等”，而是持有同一个统计对象；反思产生的 forbidden 也能阻止失败边重新进入树。


In [ ]:
def heuristic_priors(state, forbidden=frozenset()):
    if terminal_outcome(state)[0]:
        return {}
    candidates = [action for action in ACTIONS if (state_key(state), action) not in forbidden]
    if not candidates:
        return {}
    distances = []
    for action in candidates:
        candidate, _, _ = transition(state, action)
        distances.append(abs(TARGET - candidate.value))
    logits = np.array([-distance for distance in distances], dtype=float)
    probabilities = np.exp(logits - logits.max()); probabilities /= probabilities.sum()
    return dict(zip(candidates, probabilities))

def expand(node, transpositions, forbidden=frozenset()):
    if node.terminal or node.children:
        return
    transpositions.setdefault(state_key(node.state), node.stats)
    for action, prior in heuristic_priors(node.state, forbidden).items():
        state, _, _ = transition(node.state, action)
        stats = transpositions.setdefault(state_key(state), SharedStats())
        node.children[action] = Node(state, node, action, float(prior), stats)

# +1 与 *2 都从 (1,0) 到 (2,1)，因此真实共享 table 中同一个统计对象。
root = Node(start)
transpositions = {state_key(start): root.stats}
expand(root, transpositions)
assert set(root.children) == set(ACTIONS)
assert math.isclose(sum(child.prior for child in root.children.values()), 1.0)
assert all(child.parent is root for child in root.children.values())
assert root.children["+1"].state == root.children["*2"].state == State(2, 1)
assert root.children["+1"].stats is root.children["*2"].stats is transpositions[(2, 1)]

terminal_node = Node(State(TARGET, 2))
expand(terminal_node, {state_key(terminal_node.state): terminal_node.stats})
assert terminal_node.terminal and terminal_node.children == {}


## 4. Selection、reflection 与 value：失败约束进入下一次选择

Selection 每一层都过滤 `(state_key(parent), action)` forbidden，再在剩余边上比较 PUCT。Reflection 只把真实终态失败转成结构化约束；终态 value 始终使用环境 reward，不能被距离启发式覆盖。


In [ ]:
def reflection_from_failure(state, action, next_state):
    done, reward, reason = terminal_outcome(next_state)
    if not done or reward >= 0:
        return {"forbid": None, "reason": "not_failure"}
    return {"forbid": (state_key(state), action), "reason": reason}

def selectable_children(node, forbidden):
    return [child for action, child in node.children.items() if (state_key(node.state), action) not in forbidden]

def select_leaf(root, c=1.5, forbidden=frozenset()):
    validate_exploration(c)
    node, path = root, [root]
    while node.children and not node.terminal:
        candidates = selectable_children(node, forbidden)
        if not candidates:
            break
        node = max(candidates, key=lambda child: (puct(path[-1], child, c), child.action))
        path.append(node)
    return node, path

def leaf_value(node):
    if node.terminal:
        return node.reward
    return 1.0 - min(abs(TARGET - node.state.value) / TARGET, 1.0)

# selection 返回连续路径；terminal/reward 均由 state 推导，调用者无法伪造。
leaf, path = select_leaf(root)
assert path[0] is root and path[-1] is leaf
assert -1 <= leaf_value(leaf) <= 1
assert leaf_value(Node(State(TARGET, 2))) == 1.0
assert leaf_value(Node(State(MAX_VALUE + 1, 2))) == -1.0


## 5. Backpropagation：一条 simulation 给路径每个节点加一次访问

环境是单 Agent 同目标，回传同一 return；对抗游戏才交替符号。访问次数和值必须成对更新，取消/异常 simulation 不得半更新。


In [ ]:
def backpropagate(path, value):
    if not math.isfinite(value):
        raise ValueError("回传 value 必须有限")
    # transposition 或 cycle 可能让同一 stats 在路径出现多次；每次 simulation 只更新一次。
    updated = set()
    for node in path:
        identity = id(node.stats)
        if identity in updated:
            continue
        node.stats.visits += 1
        node.stats.value_sum += value
        updated.add(identity)

# 一次回传使路径上的不同共享统计各 +1，真正非路径统计保持不变。
unique_path_stats = {id(node.stats): node.stats.visits for node in path}
untouched = next(child for child in root.children.values() if child.stats is not path[-1].stats)
untouched_before = untouched.visits
backpropagate(path, leaf_value(leaf))
assert all(stats.visits == unique_path_stats[id(stats)] + 1 for stats in {id(node.stats): node.stats for node in path}.values())
assert untouched.visits == untouched_before
assert all(math.isfinite(node.q) for node in path)

# 同一 stats 通过两个 alias 出现在一条路径时仍只增加一次，而不是双计数。
alias = Node(path[-1].state, action="alias", stats=path[-1].stats)
alias_before = alias.visits
backpropagate([path[-1], alias], 0.25)
assert alias.visits == alias_before + 1 and path[-1].visits == alias.visits


## 6. 有界 LATS 搜索：参数 fail-closed，最终动作按 root visits/Q

每次 simulation 执行 selection、带 transposition/forbidden 的 expansion、评估与原子回传。失败 reflection 立即加入集合并影响之后的 selection。搜索结束不从“任意成功轨迹”挑计划，而是按 root child 的 visit count、Q、prior 依次决策。


In [ ]:
def validate_search_config(simulations, c):
    if type(simulations) is not int or simulations <= 0:
        raise ValueError("simulations 必须为正整数")
    validate_exploration(c)

def run_search(root_state, simulations=80, c=1.5):
    validate_search_config(simulations, c)
    root = Node(root_state)
    transpositions = {state_key(root_state): root.stats}
    forbidden, reflections = set(), []
    for _ in range(simulations):
        leaf, path = select_leaf(root, c, forbidden)
        if not leaf.terminal:
            expand(leaf, transpositions, forbidden)
            candidates = selectable_children(leaf, forbidden)
            if candidates:
                successful = [child for child in candidates if child.terminal and child.reward > 0]
                chosen = max(successful or candidates, key=lambda child: (child.prior, child.action))
                path.append(chosen); leaf = chosen
        value = leaf_value(leaf)
        backpropagate(path, value)
        if leaf.terminal and leaf.reward < 0 and leaf.parent is not None:
            reflection = reflection_from_failure(leaf.parent.state, leaf.action, leaf.state)
            if reflection["forbid"] is not None:
                forbidden.add(reflection["forbid"]); reflections.append(reflection)
    return root, transpositions, forbidden, reflections

def choose_root_action(root, forbidden=frozenset()):
    if root.terminal:
        raise RuntimeError("终态没有可提交 root action")
    candidates = selectable_children(root, forbidden)
    if not candidates:
        raise RuntimeError("root 没有获准动作")
    chosen = max(candidates, key=lambda child: (child.visits, child.q, child.prior, child.action))
    return chosen.action, chosen

# 固定预算严格执行 80 次 root 更新；最终动作等于显式 visits/Q 排序结果。
search_root, search_table, learned_forbidden, reflections = run_search(start, simulations=80, c=1.5)
chosen_action, chosen_child = choose_root_action(search_root, learned_forbidden)
manual_choice = max(selectable_children(search_root, learned_forbidden), key=lambda child: (child.visits, child.q, child.prior, child.action))
assert search_root.visits == 80
assert chosen_child is manual_choice and chosen_action == manual_choice.action
assert state_key(chosen_child.state) in search_table

invalid_search_config = 0
for bad_simulations, bad_c in ((0, 1.5), (-1, 1.5), (True, 1.5), (5, 0), (5, float("nan"))):
    try:
        run_search(start, bad_simulations, bad_c)
    except ValueError:
        invalid_search_config += 1
assert invalid_search_config == 5


## 7. Transposition 与 reflection 回归：共享和禁选都要可观察

仅计算相同 key 不算复用；两个 Node 必须持有同一 `SharedStats`。同样，仅生成 reflection 文本也不算约束；把失败边设成最高 Q，加入 forbidden 后再次 selection 必须不再经过该边。


In [ ]:
# transposition 的一侧回传会改变另一 alias 的 visits/Q，且 steps 不同绝不合并。
left_alias, right_alias = root.children["+1"], root.children["*2"]
right_before = right_alias.visits
backpropagate([left_alias], 0.75)
assert left_alias.stats is right_alias.stats
assert right_alias.visits == right_before + 1 and right_alias.q == left_alias.q
assert state_key(State(4, 1)) != state_key(State(4, 3))

# 构造一个故意高 Q 的 overflow 边；reflection 加入前会选它，加入后下一次 selection 必须绕开。
reflection_root = Node(State(8, 1))
reflection_table = {state_key(reflection_root.state): reflection_root.stats}
expand(reflection_root, reflection_table)
reflection_root.stats.visits = 10
overflow_child = reflection_root.children["*2"]
overflow_child.stats.visits, overflow_child.stats.value_sum = 1, 2.0
_, before_forbid_path = select_leaf(reflection_root, c=0.01, forbidden=set())
reflection = reflection_from_failure(reflection_root.state, "*2", overflow_child.state)
forbidden_probe = {reflection["forbid"]}
_, after_forbid_path = select_leaf(reflection_root, c=0.01, forbidden=forbidden_probe)
assert before_forbid_path[1].action == "*2"
assert reflection == {"forbid": ((8, 1), "*2"), "reason": "overflow"}
assert after_forbid_path[1].action != "*2"

# forbidden 在 expansion 前已存在时，失败边甚至不会被重新创建。
fresh_root = Node(State(8, 1))
expand(fresh_root, {state_key(fresh_root.state): fresh_root.stats}, forbidden_probe)
assert "*2" not in fresh_root.children


## 8. 纯 simulation 与安全 commit：获批计划、一次性消费、逐步重新鉴权

规划函数只接收不可变 state，不接触真实环境。最终 plan 的每一步都来自一次新的 root visits/Q 决策；Approval 绑定起始快照、动作序列和预期中间状态。Commit 前校验摘要，只消费一次 approval，并在每一步读取当前 state、比对快照、重新调用 authorizer 后才执行副作用。


In [ ]:
def propose_plan(root_state, simulations=80, c=1.5):
    state, actions, decisions = root_state, [], []
    for _ in range(MAX_STEPS - root_state.steps):
        done, reward, _ = terminal_outcome(state)
        if done:
            if reward > 0:
                return tuple(actions), tuple(decisions)
            raise RuntimeError("搜索到达失败终态")
        tree, _, forbidden, _ = run_search(state, simulations, c)
        action, child = choose_root_action(tree, forbidden)
        decisions.append((state_key(state), action, child.visits, child.q))
        actions.append(action)
        state, _, _ = transition(state, action)
    done, reward, _ = terminal_outcome(state)
    if not done or reward <= 0:
        raise RuntimeError("root 决策未在预算内形成成功计划")
    return tuple(actions), tuple(decisions)

@dataclass(frozen=True)
class Approval:
    plan_id: str
    start_state: State
    actions: tuple
    expected_states: tuple

def approval_id(start_state, actions, expected_states):
    payload = {"start": asdict(start_state), "actions": actions, "expected": [asdict(state) for state in expected_states]}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def approve_successful_plan(start_state, actions):
    state, expected = start_state, []
    for action in actions:
        expected.append(state)
        state, _, _ = transition(state, action)
    done, reward, _ = terminal_outcome(state)
    if not actions or not done or reward <= 0:
        raise ValueError("只能批准可重放的成功计划")
    expected = tuple(expected); actions = tuple(actions)
    return Approval(approval_id(start_state, actions, expected), start_state, actions, expected)

@dataclass
class LiveEnvironment:
    state: State
    applied_actions: list = field(default_factory=list)
    approved_plan_ids: set = field(default_factory=set)
    consumed_plan_ids: set = field(default_factory=set)
    completed_plan_ids: set = field(default_factory=set)

def commit_approved_plan(environment, approval, authorizer):
    if not isinstance(approval, Approval):
        raise PermissionError("缺少结构化 Approval")
    expected_id = approval_id(approval.start_state, approval.actions, approval.expected_states)
    if approval.plan_id != expected_id:
        raise PermissionError("Approval 摘要不匹配")
    if approval.plan_id not in environment.approved_plan_ids:
        raise PermissionError("计划未出现在真实批准集合")
    if approval.plan_id in environment.consumed_plan_ids:
        raise RuntimeError("Approval 已消费，禁止重复 commit")
    if environment.state != approval.start_state:
        raise RuntimeError("真实状态已偏离获批起点")
    if not approval.actions or len(approval.actions) != len(approval.expected_states):
        raise PermissionError("Approval 动作与预期状态长度不一致")
    replay = approval.start_state
    for action, expected_state in zip(approval.actions, approval.expected_states):
        if replay != expected_state:
            raise PermissionError("Approval 中间状态不可重放")
        replay, _, _ = transition(replay, action)
    if terminal_outcome(replay)[:2] != (True, 1.0):
        raise PermissionError("Approval 未绑定成功终态")
    environment.consumed_plan_ids.add(approval.plan_id)
    for step, (action, expected_state) in enumerate(zip(approval.actions, approval.expected_states)):
        observed = environment.state
        if observed != expected_state:
            raise RuntimeError("逐步观察发现状态漂移，停止执行")
        if not authorizer(step, observed, action, approval):
            raise PermissionError("当前步骤重新鉴权失败")
        next_state, _, _ = transition(observed, action)
        environment.state = next_state
        environment.applied_actions.append(action)
    environment.completed_plan_ids.add(approval.plan_id)
    return environment.state

@dataclass(frozen=True)
class LATSArtifact:
    environment: str
    proposal: str
    value_model: str
    exploration_c: float
    simulations: int
    max_depth: int

def artifact_hash(artifact):
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()

# simulation 只读 live snapshot；每一步 root 决策留下 visits/Q，而真实副作用仍为零。
live = LiveEnvironment(start)
live_before = (live.state, tuple(live.applied_actions), set(live.approved_plan_ids), set(live.consumed_plan_ids))
proposed_actions, root_decisions = propose_plan(live.state, simulations=80, c=1.5)
assert (live.state, tuple(live.applied_actions), live.approved_plan_ids, live.consumed_plan_ids) == live_before
assert len(root_decisions) == len(proposed_actions) and all(visits > 0 and math.isfinite(q) for _, _, visits, q in root_decisions)

approval = approve_successful_plan(start, proposed_actions)
unregistered_rejected = False
try:
    commit_approved_plan(live, approval, lambda *args: True)
except PermissionError:
    unregistered_rejected = True
assert unregistered_rejected and live.state == start and live.applied_actions == []

live.approved_plan_ids.add(approval.plan_id)
tampered = Approval("bad-digest", approval.start_state, approval.actions, approval.expected_states)
tampered_rejected = False
try:
    commit_approved_plan(live, tampered, lambda *args: True)
except PermissionError:
    tampered_rejected = True
assert tampered_rejected and live.state == start and live.applied_actions == []

# 合法 approval 每步重新鉴权一次；完成后再次提交被拒且没有重复副作用。
authorization_calls = []
def step_authorizer(step, observed, action, approved):
    authorization_calls.append((step, observed, action, approved.plan_id))
    return observed == approved.expected_states[step] and action == approved.actions[step]

committed_state = commit_approved_plan(live, approval, step_authorizer)
applied_once = tuple(live.applied_actions)
duplicate_rejected = False
try:
    commit_approved_plan(live, approval, step_authorizer)
except RuntimeError:
    duplicate_rejected = True
assert committed_state.value == TARGET and terminal_outcome(committed_state)[:2] == (True, 1.0)
assert len(authorization_calls) == len(approval.actions) == len(live.applied_actions)
assert all(call[1] == approval.expected_states[call[0]] for call in authorization_calls)
assert duplicate_rejected and tuple(live.applied_actions) == applied_once
assert approval.plan_id in live.completed_plan_ids

# 中途鉴权拒绝只执行此前已逐步获准的动作；approval 已消费，重试不会复制副作用。
denied_live = LiveEnvironment(start, approved_plan_ids={approval.plan_id})
denied_calls = []
def deny_second_step(step, observed, action, approved):
    denied_calls.append((step, observed, action))
    return step == 0

midway_rejected = False
try:
    commit_approved_plan(denied_live, approval, deny_second_step)
except PermissionError:
    midway_rejected = True
denied_applied_once = tuple(denied_live.applied_actions)
retry_rejected = False
try:
    commit_approved_plan(denied_live, approval, lambda *args: True)
except RuntimeError:
    retry_rejected = True
assert midway_rejected and len(denied_calls) == 2 and len(denied_applied_once) == 1
assert retry_rejected and tuple(denied_live.applied_actions) == denied_applied_once
assert approval.plan_id in denied_live.consumed_plan_ids and approval.plan_id not in denied_live.completed_plan_ids

# 制品绑定实际搜索常数与预算；模拟轨迹仍显式标记 simulated。
simulated_actions = [{"action": action, "mode": "simulated"} for action in proposed_actions]
artifact = LATSArtifact("number-env-v2", "root-visits-q-v2", "distance-v1", 1.5, 80, MAX_STEPS)
digest = artifact_hash(artifact)
assert all(item["mode"] == "simulated" for item in simulated_actions)
assert artifact.simulations > 0 and artifact.max_depth == MAX_STEPS
assert digest != artifact_hash(LATSArtifact("number-env-v3", artifact.proposal, artifact.value_model, 1.5, 80, MAX_STEPS))


## 面试收束：Agent/RAG 的算法只是控制面的一部分

推荐回答顺序是：任务目标和失败代价、状态/事件/证据合同、决策公式、可执行反例、离线与在线指标、权限和版本。受控环境只能证明状态机和数值关系，不能冒充开放网络、真实用户或真实模型结果。生产系统还要处理并发、超时、幂等、恶意内容、隐私、审计、灰度与回滚。

遇到追问时，主动区分模型判断与确定 verifier、计划与真实副作用、原始 observation 与 belief/memory、召回质量与生成归因，以及多尝试成功率与单次可靠性。
